# Fine-tuning `cross-encoder/ms-marco-MiniLM-L-4-v2` on NFCorpus

End-to-end notebook for fine-tuning a MiniLM cross-encoder as the reranking stage
of a two-stage medical retrieval pipeline.

- **Base model:** [`cross-encoder/ms-marco-MiniLM-L-4-v2`](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-4-v2)
- **Dataset:** [NFCorpus](https://github.com/beir-cellar/beir) (medical IR, BEIR benchmark)
- **First-stage retriever:** `hkunlp/instructor-xl` bi-encoder + FAISS

In [ ]:
!pip install -q torch sentence-transformers beir pytrec_eval faiss-cpu InstructorEmbedding

In [ ]:
import os, sys, pickle, csv, random
from typing import Dict, List, Tuple

import numpy as np
import torch
from torch.utils.data import DataLoader

import faiss
import pytrec_eval
from beir.datasets.data_loader import GenericDataLoader
from sentence_transformers import CrossEncoder, InputExample

sys.path.insert(0, '../src')
from data import (
    download_nfcorpus, load_nfcorpus_split, build_faiss_index,
    mine_hard_negatives, build_training_pairs, shuffle_samples,
)
from evaluate import evaluate
from rerank import rerank

## 1 — Download NFCorpus

Pulls the BEIR release of NFCorpus and extracts it under `data/`.

In [ ]:
nfcorpus_dir = download_nfcorpus('../data')
print('NFCorpus at:', nfcorpus_dir)

In [ ]:
train_corpus, train_queries, train_qrels = load_nfcorpus_split(nfcorpus_dir, 'train')
test_corpus,  test_queries,  test_qrels  = load_nfcorpus_split(nfcorpus_dir, 'test')
print(f'train: {len(train_corpus)} docs / {len(train_queries)} queries')
print(f'test:  {len(test_corpus)} docs / {len(test_queries)} queries')

## 2 — First-stage embeddings + FAISS index

Encode the corpus and queries with the Instructor bi-encoder. (Skip this cell if you
already have `corpus_embeddings.pkl` and `query_embeddings.pkl` saved.)

In [ ]:
from InstructorEmbedding import INSTRUCTOR

model = INSTRUCTOR('hkunlp/instructor-xl')

CORPUS_INST = 'Represent the medical document for retrieval:'
QUERY_INST  = 'Represent the medical query for retrieving relevant documents:'

corpus_inputs = [[CORPUS_INST, c.get('title', '') + '\n' + c.get('text', '')] for c in train_corpus.values()]
corpus_vecs   = model.encode(corpus_inputs).astype(np.float32)
corpus_embeddings = dict(zip(train_corpus.keys(), corpus_vecs))

query_inputs  = [[QUERY_INST, q] for q in {**train_queries, **test_queries}.values()]
query_vecs    = model.encode(query_inputs).astype(np.float32)
query_embeddings = dict(zip({**train_queries, **test_queries}.keys(), query_vecs))

os.makedirs('../data/embeddings', exist_ok=True)
with open('../data/embeddings/corpus_embeddings.pkl', 'wb') as f: pickle.dump(corpus_embeddings, f)
with open('../data/embeddings/query_embeddings.pkl',  'wb') as f: pickle.dump(query_embeddings,  f)

In [ ]:
index, id_map = build_faiss_index(corpus_embeddings)
print(f'{index.ntotal} documents indexed')

## 3 — Baseline retrieval (bi-encoder only)

Sanity check that the first-stage retriever is reasonable before we start reranking.

In [ ]:
k_eval = [1, 3, 5, 10, 100]
test_results = mine_hard_negatives(test_qrels, query_embeddings, index, id_map, top_k=100)
# scores are 1 - L2_distance — only the order matters for evaluate()
ndcg, mp, recall, prec = evaluate(test_qrels, test_results, k_eval)
print('bi-encoder only:'); print(ndcg); print(mp); print(recall); print(prec)

## 4 — Build training pairs with hard negatives

In [ ]:
train_results = mine_hard_negatives(train_qrels, query_embeddings, index, id_map, top_k=300)
train_samples = build_training_pairs(train_corpus, train_queries, train_qrels, train_results)
train_samples = shuffle_samples(train_samples)
print(f'{len(train_samples)} labelled (query, doc) pairs')

## 5 — Fine-tune the cross-encoder in chunks

After each 16384-pair chunk we re-evaluate on the test split and save the model.

In [ ]:
torch.cuda.empty_cache()
base_model = 'cross-encoder/ms-marco-MiniLM-L-4-v2'
ce = CrossEncoder(base_model, num_labels=1)

print('before fine-tuning:')
print(evaluate(test_qrels, rerank(ce, test_corpus, test_queries, test_results, 100), k_eval))

output_path = '../models/ms-marco-MiniLM-L-4-v2-nfcorpus'
os.makedirs(output_path, exist_ok=True)

steps_per_chunk = 16384
batch_size      = 164
lr              = 5e-6
warmup_steps    = 5000

n = len(train_samples)
for i in range((n + steps_per_chunk - 1) // steps_per_chunk):
    chunk = train_samples[i*steps_per_chunk : min((i+1)*steps_per_chunk, n)]
    loader = DataLoader(chunk, shuffle=True, batch_size=batch_size)
    torch.cuda.empty_cache()
    ce.fit(
        train_dataloader=loader,
        optimizer_params={'lr': lr},
        epochs=1,
        warmup_steps=warmup_steps,
        output_path=output_path,
        use_amp=True,
    )
    ce.save(output_path)
    metrics = evaluate(test_qrels, rerank(ce, test_corpus, test_queries, test_results, 100), k_eval)
    print(f'after chunk {i+1}: {metrics}')

## 6 — Use the fine-tuned model

In [ ]:
query = 'diabetes treatment'
articles = [
    'Type 1 and 2 diabetes mellitus: A review on current cure approach and gene therapy as potential intervention.',
    'Diabetes mellitus and its chronic complications. Major cause of morbidity and mortality.',
    'Diagnosis and Management of Central Diabetes Insipidus in Adults.',
    'Adipsic diabetes insipidus.',
    'Nephrogenic diabetes insipidus: a comprehensive overview.',
    'Impact of Salt Intake on the Pathogenesis and Treatment of Hypertension.',
]
pairs = [[query, a] for a in articles]
scores = ce.predict(pairs)
for s, a in sorted(zip(scores, articles), key=lambda x: -x[0]):
    print(f'{s:+.4f}  {a}')